# TP 2 — Sentiment Analysis of Customer Book Reviews

**NLP Module — Machine Learning Training**

After TP1 (preprocessing + Bag of Words / TF-IDF), we move on to a real application: automatically guessing whether a **customer review** of a book is **positive** or **negative**, based on its text.

`Raw review → Preprocessing → Vectorization (TF-IDF) → Classification model → Positive / Negative`

This is exactly the "Model" and "Result" step of the pipeline seen in class, which we hadn't used yet in TP1.

## Learning objectives

- Build a small dataset of customer reviews (a book, a comment, a sentiment).
- Reuse the preprocessing and TF-IDF from TP1.
- Train a first classification model (Logistic Regression) to predict the sentiment of a review.
- Test the model on new reviews, including your own.
- Compute, for each book, the percentage of positive reviews among its customer comments.
- Practice through the **exercises** at the end of the notebook.

> 💡 Notebook ready for **Google Colab**. Run the cells in order (`Shift + Enter`). No external file is needed: the dataset is generated directly in the notebook.

> 🆕 **This version includes, after every block of code, a line-by-line explanation written for beginners.** Look for the cells starting with 📝.

## Step 0 — Installation and imports

In [ ]:
# These libraries are already installed by default on Google Colab.
# If needed: !pip install pandas scikit-learn

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

pd.set_option("display.max_colwidth", None)
print("Libraries imported successfully ✅")

📝 **Line-by-line explanation (for beginners)**

- `# These libraries...` and `# If needed...` : these are **comments** (they start with `#`). Python completely ignores them; they are only here to inform whoever reads the code.
- `import pandas as pd` : we import the **pandas** library, used to work with tables of data (like an Excel spreadsheet). We give it the nickname `pd` so we can type less afterward.
- `from sklearn.feature_extraction.text import TfidfVectorizer` : we import the `TfidfVectorizer` tool. It knows how to turn text into numbers, so that a mathematical model can understand it.
- `from sklearn.model_selection import train_test_split` : we import the `train_test_split` function, used to split our data into two parts: one to **train** the model, one to **test** it.
- `from sklearn.linear_model import LogisticRegression` : we import the `LogisticRegression` algorithm, the model we'll use later to predict "positive" or "negative".
- `from sklearn.metrics import accuracy_score, classification_report` : we import two tools to measure how good the model is: `accuracy_score` (the percentage of correct answers) and `classification_report` (a more detailed report).
- `pd.set_option("display.max_colwidth", None)` : we tell pandas never to cut off long text when displaying it, so we can read the full reviews.
- `print("Libraries imported successfully ✅")` : we print a simple message to confirm everything worked so far.

## Step 1 — The dataset: customer reviews of books

Each row represents **a customer review of a book**, with the book title, the comment text, and its sentiment (`positive` or `negative`).

The dataset is intentionally small for this exercise (40 reviews, 5 books) to keep it simple and quick to work with. In a real project, we would use hundreds, even thousands, of reviews.

In [ ]:
reviews = [
    # --- Harry Potter and the Sorcerer's Stone ---
    {"book_title": "Harry Potter and the Sorcerer's Stone", "comment": "A magical book, I loved it from start to finish.", "sentiment": "positive"},
    {"book_title": "Harry Potter and the Sorcerer's Stone", "comment": "The characters are endearing, an incredible story.", "sentiment": "positive"},
    {"book_title": "Harry Potter and the Sorcerer's Stone", "comment": "Perfect for kids and adults alike, I highly recommend it.", "sentiment": "positive"},
    {"book_title": "Harry Potter and the Sorcerer's Stone", "comment": "A true masterpiece, full of magic and adventure.", "sentiment": "positive"},
    {"book_title": "Harry Potter and the Sorcerer's Stone", "comment": "Too childish, quite a disappointment for me.", "sentiment": "negative"},
    {"book_title": "Harry Potter and the Sorcerer's Stone", "comment": "The story is predictable and the pace too slow.", "sentiment": "negative"},
    {"book_title": "Harry Potter and the Sorcerer's Stone", "comment": "I didn't like it at all, very boring.", "sentiment": "negative"},
    {"book_title": "Harry Potter and the Sorcerer's Stone", "comment": "Honestly disappointing, I expected better.", "sentiment": "negative"},

    # --- The Lord of the Rings ---
    {"book_title": "The Lord of the Rings", "comment": "A masterpiece of fantasy, an incredible universe.", "sentiment": "positive"},
    {"book_title": "The Lord of the Rings", "comment": "I loved every page, a magnificent story.", "sentiment": "positive"},
    {"book_title": "The Lord of the Rings", "comment": "A classic that I highly recommend to everyone.", "sentiment": "positive"},
    {"book_title": "The Lord of the Rings", "comment": "The characters are endearing and the adventure incredible.", "sentiment": "positive"},
    {"book_title": "The Lord of the Rings", "comment": "Too long, the descriptions bored me.", "sentiment": "negative"},
    {"book_title": "The Lord of the Rings", "comment": "Hard to follow, quite disappointing honestly.", "sentiment": "negative"},
    {"book_title": "The Lord of the Rings", "comment": "The writing style is heavy, I didn't like it.", "sentiment": "negative"},
    {"book_title": "The Lord of the Rings", "comment": "Disappointing and predictable, I gave up before the end.", "sentiment": "negative"},

    # --- Dune ---
    {"book_title": "Dune", "comment": "An incredible saga, I loved this magnificent universe.", "sentiment": "positive"},
    {"book_title": "Dune", "comment": "Excellent and immersive, a true masterpiece of science fiction.", "sentiment": "positive"},
    {"book_title": "Dune", "comment": "I highly recommend it, the characters are endearing.", "sentiment": "positive"},
    {"book_title": "Dune", "comment": "Fascinating from start to finish, I loved it.", "sentiment": "positive"},
    {"book_title": "Dune", "comment": "Too confusing, quite a disappointment for a cult classic.", "sentiment": "negative"},
    {"book_title": "Dune", "comment": "I got lost, the story is too slow and boring.", "sentiment": "negative"},
    {"book_title": "Dune", "comment": "Honestly disappointing, I didn't like it at all.", "sentiment": "negative"},
    {"book_title": "Dune", "comment": "Predictable and heavy, I expected better.", "sentiment": "negative"},

    # --- The Da Vinci Code ---
    {"book_title": "The Da Vinci Code", "comment": "An incredible thriller, I loved the suspense.", "sentiment": "positive"},
    {"book_title": "The Da Vinci Code", "comment": "Full of twists, a masterpiece of the genre.", "sentiment": "positive"},
    {"book_title": "The Da Vinci Code", "comment": "I highly recommend it, a magnificent read.", "sentiment": "positive"},
    {"book_title": "The Da Vinci Code", "comment": "The characters are endearing, truly incredible.", "sentiment": "positive"},
    {"book_title": "The Da Vinci Code", "comment": "Too predictable, quite disappointing in the end.", "sentiment": "negative"},
    {"book_title": "The Da Vinci Code", "comment": "The writing is heavy and boring at times.", "sentiment": "negative"},
    {"book_title": "The Da Vinci Code", "comment": "Honestly disappointing, I didn't like it.", "sentiment": "negative"},
    {"book_title": "The Da Vinci Code", "comment": "Repetitive and predictable, I expected better.", "sentiment": "negative"},

    # --- Pride and Prejudice ---
    {"book_title": "Pride and Prejudice", "comment": "A magnificent novel, I loved this love story.", "sentiment": "positive"},
    {"book_title": "Pride and Prejudice", "comment": "An incredible classic that I highly recommend.", "sentiment": "positive"},
    {"book_title": "Pride and Prejudice", "comment": "The characters are endearing, a true masterpiece.", "sentiment": "positive"},
    {"book_title": "Pride and Prejudice", "comment": "I loved every moment, a magnificent read.", "sentiment": "positive"},
    {"book_title": "Pride and Prejudice", "comment": "Too slow, quite a disappointment for this classic.", "sentiment": "negative"},
    {"book_title": "Pride and Prejudice", "comment": "The old-fashioned vocabulary makes the reading boring.", "sentiment": "negative"},
    {"book_title": "Pride and Prejudice", "comment": "Honestly disappointing, I didn't like it at all.", "sentiment": "negative"},
    {"book_title": "Pride and Prejudice", "comment": "Predictable and heavy, I expected better.", "sentiment": "negative"},
]

df = pd.DataFrame(reviews)
df.to_csv("book_reviews.csv", index=False, encoding="utf-8")

print(f"Dataset created: {len(df)} reviews for {df['book_title'].nunique()} books")
df.head(10)

📝 **Line-by-line explanation (for beginners)**

- `reviews = [ ... ]` : we create a **list** called `reviews`. A list is a sequence of several items, here inside square brackets `[ ]`.
- Each item in the list, inside curly braces `{ }`, is a **dictionary**: it represents **one single customer review**, with three pieces of information attached to a name (a "key"): `"book_title"` (the book concerned), `"comment"` (the review text) and `"sentiment"` (`"positive"` or `"negative"`).
- The lines starting with `# ---` are just comments to visually separate the reviews book by book (Harry Potter, The Lord of the Rings, Dune, etc.). They have no effect on the code.
- `df = pd.DataFrame(reviews)` : we turn the `reviews` list into a real **table** using pandas (a "DataFrame"), with one column per piece of information (book title, comment, sentiment) and one row per review — like an Excel table.
- `df.to_csv("book_reviews.csv", index=False, encoding="utf-8")` : we save this table into a file called `book_reviews.csv`, so it can be reused later without retyping everything. `index=False` avoids adding a useless column with the row numbers.
- `print(f"Dataset created: {len(df)} reviews for {df['book_title'].nunique()} books")` : we display a summary. `len(df)` counts the total number of rows (so, of reviews), and `df['book_title'].nunique()` counts the number of **different** book titles.
- `df.head(10)` : we display the first 10 rows of the table, to quickly check that everything looks correct.

### Use your own review file

If you have a real `book_reviews.csv` file (with columns `book_title`, `comment`, `sentiment`), upload it to Colab (📁 icon) then run this cell instead.

In [ ]:
# Uncomment if you are using your own file
# df = pd.read_csv("book_reviews.csv")
# df.head()

📝 **Line-by-line explanation (for beginners)**

- Both lines start with `#` : they are **commented out**, meaning disabled. They do not run.
- They serve as an example: if you have your own `book_reviews.csv` file, just remove the `#` in front of each line so that `pd.read_csv("book_reviews.csv")` loads your data instead of the one created just above, and `df.head()` shows a preview of it.

## Step 2 — Quick exploration

Let's look at the positive / negative breakdown, overall and per book.

In [ ]:
print("Overall sentiment breakdown:")
print(df["sentiment"].value_counts())
print()

print("Number of reviews per book:")
print(df["book_title"].value_counts())

📝 **Line-by-line explanation (for beginners)**

- `print("Overall sentiment breakdown:")` : prints a title to introduce what comes next.
- `df["sentiment"]` : selects only the `"sentiment"` column of the `df` table.
- `.value_counts()` : counts how many times each different value appears in that column (so, how many "positive" reviews and how many "negative" ones).
- The surrounding `print(...)` : displays that result on screen.
- `print()` (with nothing inside) : simply prints an empty line, to make the output easier to read.
- The last two lines do the same thing, but on the `"book_title"` column: they display how many reviews we have for each book.

In [ ]:
df.groupby(["book_title", "sentiment"]).size().unstack(fill_value=0)

📝 **Line-by-line explanation (for beginners)**

- `df.groupby(["book_title", "sentiment"])` : we group the rows of the table by "book + sentiment" pairs (for example, all the positive reviews of Dune together, all the negative reviews of Dune together, and so on).
- `.size()` : for each group created, we count how many rows it contains.
- `.unstack(fill_value=0)` : we reshape this result into a proper table, with one book per row and one "positive" column / one "negative" column. `fill_value=0` replaces with 0 any cell where there would be no review at all (instead of leaving it empty).

## Step 3 — Text preprocessing

We reuse the simple cleaning from TP1: lowercasing, punctuation removal, tokenization, stopword removal.

In [ ]:
stopwords_en = {
    "the", "a", "an", "of", "and", "is", "i", "to",
    "it", "this", "that", "for", "with", "my", "in", "on",
}

def clean_text(text):
    """Cleans a review: lowercase, punctuation removal, tokenization, stopwords."""
    text = text.lower()
    for char in "!?.,;:\"'":
        text = text.replace(char, " ")
    words = text.split()
    words = [w for w in words if w not in stopwords_en]
    return " ".join(words)

df["clean_comment"] = df["comment"].apply(clean_text)
df[["comment", "clean_comment"]].head(6)

📝 **Line-by-line explanation (for beginners)**

- `stopwords_en = { ... }` : we create a **set** (inside curly braces `{ }`) of very common English words (the, a, of, and...) that don't carry useful information about the sentiment of a review. We want to ignore them.
- `def clean_text(text):` : we **define a function** called `clean_text`. A function is a small, reusable block of code; here it takes a `text` as input.
- `\"\"\"Cleans a review: ...\"\"\"` : this is a **docstring**, a sentence describing what the function does (it has no effect on the result, it's just documentation).
- `text = text.lower()` : we turn the whole text into lowercase, so that "Book" and "book" are treated the same way.
- `for char in "!?.,;:\"'":` : we go through each punctuation character in this string, one by one.
- `text = text.replace(char, " ")` : on each loop iteration, we replace that punctuation character with a space in the text.
- `words = text.split()` : we split the text (which no longer has punctuation) into a list of separate words, based on the spaces.
- `words = [w for w in words if w not in stopwords_en]` : we rebuild the list of words, keeping only the ones that are **not** in the stopwords list defined above. This is called a "list comprehension": a compact way of writing a loop that filters a list.
- `return " ".join(words)` : we glue all the remaining words back together into a single sentence, separated by spaces, and return this cleaned text as the function's result.
- `df["clean_comment"] = df["comment"].apply(clean_text)` : we apply the `clean_text` function to **every** comment in the `"comment"` column, and store the results in a new column `"clean_comment"`.
- `df[["comment", "clean_comment"]].head(6)` : we display the first 6 rows, with the original comment and its cleaned version side by side, to visually compare the cleaning result.

## Step 4 — TF-IDF vectorization

As in TP1, we transform each cleaned review into a vector of numbers using `TfidfVectorizer`.

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["clean_comment"])
y = df["sentiment"]

print("Shape of the TF-IDF matrix:", X.shape)
print(f"→ {X.shape[0]} reviews, {X.shape[1]} unique words in the vocabulary")

📝 **Line-by-line explanation (for beginners)**

- `vectorizer = TfidfVectorizer()` : we create a `TfidfVectorizer` object, the tool that will know how to turn text into numbers.
- `X = vectorizer.fit_transform(df["clean_comment"])` : we apply this tool to all the cleaned comments. `X` now holds every review as a **vector of numbers** (a large matrix) instead of text.
- `y = df["sentiment"]` : we set aside the `"sentiment"` column, which holds the **true answers** (positive or negative). This is what the model will have to learn to guess from `X`.
- `print("Shape of the TF-IDF matrix:", X.shape)` : `X.shape` gives the dimensions of the `X` matrix, i.e. (number of reviews, number of different words in the vocabulary).
- `print(f"→ {X.shape[0]} reviews, {X.shape[1]} unique words in the vocabulary")` : we print that same information again, in a more readable form.

## Step 5 — Train a classification model

This is what's new in this exercise: instead of comparing texts to each other (similarity), we train a **classification model** — a Logistic Regression — to recognize positive and negative reviews from their TF-IDF vectors.

We first split the dataset in two: one part to **train** the model, one part to **test** it on reviews it has never seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=44, stratify=y
)

print(f"Training reviews: {X_train.shape[0]}")
print(f"Test reviews     : {X_test.shape[0]}")

📝 **Line-by-line explanation (for beginners)**

- `X_train, X_test, y_train, y_test = train_test_split(...)` : this function automatically splits the data into **4 pieces**: the reviews (vectorized) and their sentiments for training (`X_train`, `y_train`), and the reviews and their sentiments for testing (`X_test`, `y_test`).
- `X, y` : the starting data we want to split (the vectorized reviews and their true labels).
- `test_size=0.3` : we keep 30% of the reviews for testing, so the remaining 70% is used for training.
- `random_state=44` : a "seed" number that fixes the randomness, so the split is always exactly the same if you rerun the code later (reproducibility).
- `stratify=y` : we ask to keep the same proportion of positive/negative reviews in both the training set and the test set (otherwise randomness could unbalance the two groups).
- The two `print(...)` lines simply display how many reviews end up in each group, using `.shape[0]` which gives the number of rows.

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

print("Model trained ✅")

📝 **Line-by-line explanation (for beginners)**

- `model = LogisticRegression()` : we create a Logistic Regression model. At this point it doesn't know anything yet, it hasn't been trained.
- `model.fit(X_train, y_train)` : this is **the training step**. The model looks at the training reviews (`X_train`) and their true answers (`y_train`), and learns which words, or combinations of words, are associated with a positive or negative review.
- `print("Model trained ✅")` : simple message to confirm training finished.

## Step 6 — Evaluate the model

Let's see how well the model performs (or not) on the test reviews, which it never saw during training.

In [ ]:
test_predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, test_predictions)
print(f"Model accuracy on test reviews: {accuracy:.0%}\n")

print(classification_report(y_test, test_predictions))

📝 **Line-by-line explanation (for beginners)**

- `test_predictions = model.predict(X_test)` : we ask the model to guess the sentiment of every review in the **test** set, which it never saw during training.
- `accuracy = accuracy_score(y_test, test_predictions)` : we compare the true answers (`y_test`) to the model's predictions (`test_predictions`), and compute the percentage of times they match: this is the **overall accuracy**.
- `print(f"Model accuracy on test reviews: {accuracy:.0%}\n")` : we display this result, formatted as a rounded percentage thanks to `:.0%`; the `\n` at the end adds an empty line afterward.
- `print(classification_report(y_test, test_predictions))` : we display a more detailed report, with separate statistics for the "positive" class and the "negative" class (for example, how reliable the model is on each one).

> ⚠️ With only 40 reviews, this score can vary depending on which examples are randomly picked for the test set. That's fine: the goal of this exercise is to understand the principle, not to get a perfect model (that would require hundreds/thousands of reviews).

## Step 7 — Predict the sentiment of a new review

Let's build a function that takes a comment (never seen by the model), and returns whether it's rather **positive** or **negative**, along with the model's confidence level.

In [ ]:
def predict_sentiment(comment, model=model, vectorizer=vectorizer):
    """Predicts the sentiment (positive/negative) of a comment, with the confidence score."""
    clean_comment = clean_text(comment)
    vector = vectorizer.transform([clean_comment])

    prediction = model.predict(vector)[0]
    probabilities = model.predict_proba(vector)[0]
    confidence = max(probabilities)

    print(f"Review    : {comment}")
    print(f"Sentiment : {prediction} (confidence: {confidence:.0%})")
    return prediction

📝 **Line-by-line explanation (for beginners)**

- `def predict_sentiment(comment, model=model, vectorizer=vectorizer):` : we define a new function, which takes a `comment` as text, and by default uses the `model` and `vectorizer` already trained above.
- `\"\"\"Predicts the sentiment ...\"\"\"` : the docstring, describing what the function does.
- `clean_comment = clean_text(comment)` : we reuse the `clean_text` function created in Step 3, to clean this new comment (lowercase, no punctuation, no stopwords).
- `vector = vectorizer.transform([clean_comment])` : we turn this cleaned comment into a vector of numbers, using the **same** `vectorizer` as for training. We use `.transform()` (not `.fit_transform()`) because we don't want to relearn a new vocabulary, just reuse the one already learned. The square brackets `[...]` are needed because this method expects a list of texts, even if there's only one here.
- `prediction = model.predict(vector)[0]` : we ask the model for its prediction ("positive" or "negative"). The `[0]` grabs just the result, because `.predict()` always returns a list.
- `probabilities = model.predict_proba(vector)[0]` : we retrieve the probabilities the model assigns to each class (for example 80% positive, 20% negative).
- `confidence = max(probabilities)` : we keep the highest probability, which represents how "confident" the model is in its prediction.
- The two `print(...)` lines display the original review and the prediction result in a readable way, with the confidence as a percentage.
- `return prediction` : the function returns the prediction, so it can be reused elsewhere if needed (for example, stored in a variable).

In [ ]:
# Let's test on a few made-up reviews, never seen by the model
_ = predict_sentiment("I loved this book, a magnificent story with unforgettable characters!")
print()
_ = predict_sentiment("What a disappointment, I was bored from start to finish.")

📝 **Line-by-line explanation (for beginners)**

- `_ = predict_sentiment("...")` : we call the `predict_sentiment` function created just above, with a new made-up review. The name `_` (underscore) is a Python convention meaning "I won't reuse this value"; we use it here because we only care about what the function prints, not about what it returns.
- `print()` : prints an empty line, to visually separate the two results.
- The last line does exactly the same thing, but with a made-up negative review, to see whether the model correctly recognizes it as such.

In [ ]:
# And on a review of your own!
_ = predict_sentiment("This book is truly incredible, I recommend it to everyone.")

📝 **Line-by-line explanation (for beginners)**

- Same idea as the previous cell: we call `predict_sentiment` with a third review. You can replace the text between quotes with your own review to test the model yourself.

## Step 8 — The percentage of positive reviews, per book

This is often the real business need: not just classifying a single review, but **summarizing the overall customer opinion for each book**.

In [ ]:
summary_by_book = (
    df.groupby("book_title")["sentiment"]
    .apply(lambda s: (s == "positive").mean())
    .sort_values(ascending=False)
    .rename("percentage_positive_reviews")
    .apply(lambda x: f"{x:.0%}")
)

summary_by_book

📝 **Line-by-line explanation (for beginners)**

- `df.groupby("book_title")["sentiment"]` : we group all the rows of the table by book, and for each group, we look only at the `"sentiment"` column.
- `.apply(lambda s: (s == "positive").mean())` : for each book, `s` represents the list of its sentiments. `s == "positive"` turns each sentiment into `True` (if positive) or `False` (otherwise), and `.mean()` computes the average of these `True`/`False` values (where `True` counts as 1 and `False` as 0): this gives exactly the **proportion** of positive reviews for that book. A `lambda` is a small function written on a single line, without giving it a name with `def`.
- `.sort_values(ascending=False)` : we sort the books from **most positive** to **least positive** (`ascending=False` means "not in increasing order", so decreasing).
- `.rename("percentage_positive_reviews")` : we give this series of results a clear name.
- `.apply(lambda x: f"{x:.0%}")` : we turn each decimal number (for example 0.75) into readable text formatted as a percentage (for example "75%").
- `summary_by_book` : as the last line of a cell, just writing a variable's name automatically displays it in Jupyter/Colab — no need for `print()`.

Here we compute the percentage from the **true labels** in the dataset. In a real-world scenario, we would do the same thing using the **model's predictions**, on new reviews that haven't been manually labeled yet — which is exactly the point of Exercise 5.

## 🎯 Exercises — Your turn!

### Exercise 1 — Test your own reviews

Write 3 reviews of your own invention about a book (2 positive, 1 negative, or vice versa) and pass them into `predict_sentiment()`. Does the model get any of them wrong?

In [ ]:
# TODO Exercise 1
# _ = predict_sentiment("...")
# _ = predict_sentiment("...")
# _ = predict_sentiment("...")

📝 **Line-by-line explanation (for beginners)**

- All these lines are commented out (`#`), so nothing runs yet: that's normal, it's up to you to complete them.
- For each line, you'll need to: remove the leading `#`, then replace `"..."` with a real review you invent, keeping the quotes around the text.
- Each line calls the `predict_sentiment()` function created in Step 7, which will automatically print the model's prediction (positive/negative) and its confidence level.

### Exercise 2 — Change the test set size

Redo the split with `test_size=0.2` then `test_size=0.5` (Step 5), retraining the model (Steps 5 and 6) each time. How does the accuracy change? Why isn't a test set that's too small very reliable?

In [ ]:
# TODO Exercise 2
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# model.fit(X_train, y_train)
# print(accuracy_score(y_test, model.predict(X_test)))

📝 **Line-by-line explanation (for beginners)**

- Line 1 (once uncommented) : redoes the train/test split like in Step 5, but changing `test_size` (here 0.2, so 20% for testing instead of 30%).
- Line 2 : retrains the model on this new split, with `.fit()`, exactly as in Step 5.
- Line 3 : recomputes and directly prints the accuracy (`accuracy_score`) on this new test set, combining into a single line what was done over several lines in Step 6.
- It's up to you to copy these lines several times, changing `test_size` (0.2, then 0.5), to compare the results you get.

### Exercise 3 — Try another model: Naive Bayes

`MultinomialNB` is another classification algorithm, often used for text. Replace `LogisticRegression` with `MultinomialNB` (same training, same evaluation) and compare the accuracy obtained.

In [ ]:
# TODO Exercise 3
# from sklearn.naive_bayes import MultinomialNB
# nb_model = MultinomialNB()
# nb_model.fit(X_train, y_train)
# print(accuracy_score(y_test, nb_model.predict(X_test)))

📝 **Line-by-line explanation (for beginners)**

- Line 1 : imports a new algorithm, `MultinomialNB` (Naive Bayes), from scikit-learn — just like we did for `LogisticRegression` in Step 0.
- Line 2 : creates a new model, `nb_model`, not yet trained, based on this algorithm.
- Line 3 : trains this model on the same training data (`X_train`, `y_train`) as before, using `.fit()`.
- Line 4 : computes and prints this new model's accuracy on the test set, to compare it with the one obtained with Logistic Regression in Step 6.

### Exercise 4 — Look at the model's mistakes

Display the test reviews that the model **misclassified** (prediction different from the true label). In your opinion, why did the model get these particular reviews wrong?

In [ ]:
# TODO Exercise 4
# test_results = pd.DataFrame({
#     "comment": df.loc[y_test.index, "comment"],
#     "true_sentiment": y_test,
#     "predicted_sentiment": test_predictions,
# })
# errors = test_results[test_results["true_sentiment"] != test_results["predicted_sentiment"]]
# errors

📝 **Line-by-line explanation (for beginners)**

- `test_results = pd.DataFrame({...})` : builds a new summary table, with three columns: the original comment, the true sentiment (`y_test`), and the sentiment predicted by the model (`test_predictions`, computed in Step 6).
- `df.loc[y_test.index, "comment"]` : retrieves the original comments matching exactly the same rows as those present in `y_test` (using their row numbers, `y_test.index`).
- `errors = test_results[test_results["true_sentiment"] != test_results["predicted_sentiment"]]` : filters this table to keep only the rows where the true sentiment is **different** from the one predicted by the model (the `!=` symbol means "different from") — so, only the model's mistakes.
- `errors` : displays this filtered table, to examine the reviews the model got wrong.

### Exercise 5 (bonus) — Percentage of positive reviews from predictions

In Step 8, we computed the percentage of positive reviews per book from the **true labels**. Redo the same calculation, but this time using the **model's predictions** on the entire dataset (`model.predict(X)`), as if we didn't know the true labels. Is the result close to Step 8?

In [ ]:
# TODO Exercise 5 (bonus)
# df["predicted_sentiment"] = model.predict(X)
# predicted_summary = (
#     df.groupby("book_title")["predicted_sentiment"]
#     .apply(lambda s: (s == "positive").mean())
#     .apply(lambda x: f"{x:.0%}")
# )
# predicted_summary

📝 **Line-by-line explanation (for beginners)**

- `df["predicted_sentiment"] = model.predict(X)` : asks the model to predict the sentiment of **all** the reviews in the dataset (`X`, not just the test ones), and stores these predictions in a new column `"predicted_sentiment"`.
- The following lines reuse exactly the same logic as Step 8 (`groupby`, `apply` with a mean, then formatting as a percentage), but based on the `"predicted_sentiment"` column (the model's predictions) instead of `"sentiment"` (the true labels).
- `predicted_summary` : displays this prediction-based summary, to compare against `summary_by_book` from Step 8 and see whether the model gives a faithful picture of customers' real opinions.

## Conclusion

You have built a first end-to-end **sentiment analysis** system:

**Raw review → Preprocessing → TF-IDF → Classification model → Positive / Negative**

You now know how to:

- prepare a labeled dataset (text + sentiment),
- split the data into training / test sets to honestly evaluate a model,
- train a simple classification model on vectorized text,
- use it to predict the sentiment of new reviews,
- summarize customer opinion at the level of an entire book.

### Going further

- A larger dataset (hundreds/thousands of reviews) would produce a much more reliable model.
- A **neutral** category, in addition to positive/negative, for mixed reviews.
- More advanced models (SVM, embeddings, Transformers/BERT) to better capture nuances (irony, negation...).

Good work! 🚀